# Qwen3.8-27B → DeepSeek Harness

This notebook uses the outbound-only Supabase relay. No reverse tunnel or public Colab port is used.

In Colab **Secrets** (key icon), add `QWEN_RELAY_SECRET` and enable **Notebook access**.

For automatic Oracle VM wake-up, also add `ORACLE_WAKE_GITHUB_TOKEN` and enable **Notebook access**. Use a fine-grained token restricted to the relay repository with only the required Actions permission.

When Section 2 starts, Colab immediately creates a short heartbeat lease in Supabase. This keeps the shared Oracle VM online while Qwen is still installing/downloading/loading. The public API does **not** report Qwen ready until vLLM is actually ready.

For a network/API test with **zero GPU usage**, run **Section 1** and then **Section 3** only. For the real model, run **Section 1** and then **Section 2** on an A100 80 GB runtime. Do not run Sections 2 and 3 at the same time.


## Section 1 — Repair Colab Python packages + install/update worker
This first repairs Pillow as a clean package because some Colab images can contain mixed `PIL` files that fail with `cannot import name _Ink from PIL._typing`. Then it installs the latest worker from GitHub. If this runtime has **already** shown the `_Ink` ImportError, run Section 1, choose **Runtime → Restart session**, then continue with Section 2.


In [ ]:
%pip uninstall -y Pillow >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall Pillow
%pip install -q --upgrade --force-reinstall "git+https://github.com/Logan17de/All-testing.git#subdirectory=llm"
from PIL import ImageText
import PIL
print(f"Pillow {PIL.__version__}: OK ✅")


## Optional — capture BEFORE benchmark
If the old BF16 vLLM server is **still running in this same Colab runtime**, run this once before Section 2. It measures localhost TTFT and decode tokens/sec and saves the result under `/content/qwen_benchmark_results.json`. Section 2 also auto-detects and benchmarks a legacy running server before replacing it, so this cell is optional.


In [ ]:
import importlib
import qwen3_8_27b_supabase_colab_fast_nightly as qwen_fast
importlib.reload(qwen_fast)
qwen_fast.benchmark_running_server("before")


## Section 2 — REAL Qwen3.8-27B optimized worker (A100 80 GB)
This uses official `Qwen/Qwen3.8-27B-FP8` weights, native 262,144 context, FP8 KV cache, native MTP with 3 draft tokens, prefix caching, chunked prefill, torch.compile/CUDA Graphs, and single-user scheduling. It installs the CUDA-13 vLLM **nightly** with the Qwen3.8 gated-DeltaNet MTP fix (0.27.2+ development line) using `uv`; stable 0.27.1 is rejected for this MTP profile. On A100, the profile uses FlashInfer for FP8-KV-compatible attention and Marlin for FP8 W8A16 linear layers. The local benchmark runs before the worker is advertised ready and reports TTFT, output tok/s and MTP acceptance.


In [ ]:
import importlib
import qwen3_8_27b_supabase_colab_fast_nightly as qwen_worker
importlib.reload(qwen_worker)
qwen_worker.main()


## Section 3 — TESTING: API/relay only (NO GPU)
Use a normal CPU Colab runtime. Run **Section 1**, skip Section 2, then run this section. No Torch, vLLM, CUDA, Hugging Face model, or GPU is used. Every request received from Harness is decoded through the real relay and returned as the OpenAI-compatible assistant response **`succeed`**. Leave this cell running while testing Harness.


In [ ]:
import importlib
import qwen_supabase_test_worker as relay_test
importlib.reload(relay_test)
relay_test.main()
